# 5b — GBM TARGETS

Apply trained GBM models to Antarctic and Greenland target grids.

Outputs per region (single pass, three GBM heads):

| Variable | Description |
|---|---|
| `gbm_q05/q25/q50/q75/q95_raw` | Raw GBM predictions (Q05, Q50, Q95 direct; Q25/Q75 imputed) |
| `gbm_q05/q25/q50/q75/q95_corr` | Spline-corrected (Q50 shift applied) |
| `gbm_q05/q95_conf` | Conformal-inflated bounds |
| `gbm_iqr50/iqr90_corr/conf` | Inter-quantile ranges |
| `gbm_sigma_corr/conf` | Gaussian-equiv σ from IQR90 |

**Spline correction**: PCHIP spline fitted on calibration Q50. Applied to Q50 only.
Other quantiles shifted by the same additive offset (interval widths preserved).

**Conformal bounds**: Q05/Q95 additionally inflated by ±q_hat_gbm.

**Q25/Q75**: linearly interpolated between Q05–Q50 and Q50–Q95 respectively.

**Files written**
- `output/targets/Aq15GBMv{MODEL_VERSION}.nc`  Antarctica
- `output/targets/Kq15GBMv{MODEL_VERSION}.nc`  Greenland

Requires `output/models/gbm_artefacts.pkl` written by `4b_GBM_MODEL`.

## 0. Config patch check

In [1]:
# config.py additions needed
# Add these lines to config.py if not already present:
#
# MODEL_VERSION = "02"
#
# model_paths.update({
#     "gbm_artefacts": model_dir / "gbm_artefacts.pkl",
# })
#
# Each GBM artefact pkl is produced by 4b_GBM_MODEL for each quantile.
# The pkl must contain keys: model, gbm_models, scaler, gbm_spline, obs_sel, PARAMS
print("Config note displayed.")


Config note displayed.


## 1. Imports & constants

In [ ]:
import sys, json, pickle, warnings, datetime
import time

import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path
from pyproj import Transformer
from scipy.stats import entropy as scipy_entropy
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

sys.path.insert(0, ".")
from config import *

# (
#     obs_model, q_clip_min, q_clip_max, random_state, q_min, q_max,
#     ant_crs, grl_crs, model_dir, targets_dir, param_paths,
#     netcdf_compression_level, NETCDF_AUTHOR, NETCDF_CONVENTIONS,
#     PRED_QUANTILES, TARGET_GRIDS, MODEL_VERSION,
# )

targets_dir.mkdir(parents=True, exist_ok=True)

QUANTILES   = PRED_QUANTILES   # [0.05, 0.25, 0.50, 0.75, 0.95]
Q50_IDX     = QUANTILES.index(0.50)
GBM_LOSSES  = [0.05, 0.50, 0.95]   # quantiles with trained GBM models

print(f"quantiles : {QUANTILES}")
print(f"GBM heads : {GBM_LOSSES}")
print(f"model version : {MODEL_VERSION}")
print(f"targets_dir : {targets_dir}")


✓ config.py v4.0 | obs_model: 21 | obs_sweep: 22 | obs: 37
quantiles : [0.05, 0.25, 0.5, 0.75, 0.95]
GBM heads : [0.05, 0.5, 0.95]
model version : 0_2
targets_dir : output/targets


## 2. Helpers

In [ ]:
# Grid loader
def load_grid(grid_cfg, obs_sel, label=None):
    label   = label or grid_cfg["label"]
    x_col   = grid_cfg["x_col"]
    y_col   = grid_cfg["y_col"]
    df = pd.read_parquet(grid_cfg["parquet"])
    print(f"{label.upper():6s}  {len(df):>10,} rows")
    missing = [f for f in obs_sel if f not in df.columns]
    if missing:
        warnings.warn(f"Missing features {missing}; filling NaN")
        for f in missing:
            df[f] = np.nan
    nan_cols = {f: int(df[f].isna().sum()) for f in obs_sel if df[f].isna().any()}
    if nan_cols:
        print(f"  NaN counts: {nan_cols}")
    else:
        print(f"  All {len(obs_sel)} features OK, no NaNs")
    x_vals = np.sort(df[x_col].unique()).astype(np.float64)
    y_vals = np.sort(df[y_col].unique()).astype(np.float64)
    nx, ny  = len(x_vals), len(y_vals)
    is_reg  = len(df) == nx * ny
    if is_reg:
        dx, dy = np.diff(x_vals), np.diff(y_vals)
        print(f"  Regular {ny}x{nx}  dx={dx[0]:.0f}m  dy={dy[0]:.0f}m")
    else:
        print(f"  Irregular {len(df):,} pts  (expected {ny}x{nx})")
    return df, x_vals, y_vals, ny, nx, is_reg


def build_X(df, obs_sel, scaler):
    X_raw  = df[obs_sel].values.astype(np.float32)
    finite = np.isfinite(X_raw).all(axis=1)
    X_sc   = np.full_like(X_raw, np.nan)
    if finite.any():
        X_sc[finite] = scaler.transform(X_raw[finite]).astype(np.float32)
    print(f"  finite points: {finite.sum():,} / {len(df):,}")
    return X_sc, finite


def to_2d(vals_1d, df, x_vals, y_vals, x_col, y_col, is_reg):
    ny, nx = len(y_vals), len(x_vals)
    if is_reg:
        tmp = df[[x_col, y_col]].copy()
        tmp["v"] = vals_1d
        return tmp.pivot(index=y_col, columns=x_col, values="v").values.astype(np.float32)
    from scipy.interpolate import griddata
    gx, gy = np.meshgrid(x_vals, y_vals)
    return griddata(df[[x_col, y_col]].values, vals_1d, (gx, gy),
                    method="nearest").astype(np.float32)


def apply_spline(spline, vals):
    out = np.full(vals.shape, np.nan, dtype=np.float32)
    ok  = np.isfinite(vals)
    out[ok] = np.clip(spline(vals[ok]).astype(np.float32), q_clip_min, q_clip_max)
    return out


def shannon_H_norm(values_2d, bin_width=0.010):
    v = values_2d.ravel()
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return np.nan
    bins   = np.arange(q_clip_min, q_clip_max + bin_width, bin_width)
    counts, _ = np.histogram(v, bins=bins)
    p = counts / counts.sum()
    p = p[p > 0]
    H     = float(scipy_entropy(p))
    H_max = float(np.log(len(bins) - 1))
    return round(H / H_max, 6) if H_max > 0 else 0.0


def make_ds(arrays, x_vals, y_vals, grid_cfg, params, obs_sel, model_tag, extra=None):
    coords = {"y": y_vals, "x": x_vals}
    dvars  = {name: xr.DataArray(arr, dims=["y", "x"], coords=coords,
                                 attrs={"units": "W m-2", "long_name": name.replace("_", " ")})
              for name, arr in arrays.items()}
    ds = xr.Dataset(dvars, coords=coords,
        attrs=dict(
            title       = f"{model_tag} heat-flow prediction",
            model       = model_tag,
            version     = MODEL_VERSION,
            institution = NETCDF_AUTHOR,
            Conventions = NETCDF_CONVENTIONS,
            crs         = grid_cfg["crs"],
            epsg        = str(grid_cfg["epsg"]),
            region      = grid_cfg["label"].upper(),
            created     = datetime.datetime.utcnow().isoformat() + "Z",
            obs_sel     = json.dumps(obs_sel),
            quantiles   = json.dumps(QUANTILES),
            q_clip_min  = float(q_clip_min),
            q_clip_max  = float(q_clip_max),
            params_json = json.dumps(params, default=str),
        ))
    if extra:
        ds.attrs.update(extra)
    return ds


def save_nc(ds, path):
    enc = {v: {"zlib": True, "complevel": netcdf_compression_level, "dtype": "float32"}
           for v in ds.data_vars}
    ds.to_netcdf(path, encoding=enc)
    print(f"  Saved {path}  ({path.stat().st_size/1e6:.1f} MB)")

print("Helpers ready.")


Helpers ready.


## 3. Load GBM artefacts

In [ ]:
with open(model_dir / "gbm_artefacts.pkl", "rb") as fp:
    art = pickle.load(fp)

gbm_models = art["gbm_models"]      # {0.05: model, 0.50: model, 0.95: model}
scaler     = art["scaler"]
gbm_spline = art["gbm_spline"]      # PCHIP spline fitted on Q50 cal set
obs_sel    = art["obs_sel"]
q_hat_gbm  = art["q_hat_gbm"]       # conformal correction
PARAMS     = art["PARAMS"]
Q_LOSSES   = art["quantile_losses"] # [0.05, 0.50, 0.95]

print(f"GBM loaded  n_models={len(gbm_models)}  features={len(obs_sel)}")
print(f"obs_sel : {obs_sel}")
print(f"q_hat_gbm  = {q_hat_gbm:.6f}")


GBM loaded  n_models=3  features=21
obs_sel : ['MOHO', 'MOHO_GRAV', 'DEM', ...]
q_hat_gbm  = 0.031200


## 4. Predict & write netCDF

Loops over `TARGET_GRIDS` (ant, grl).  For each grid:
1. Load parquet, verify features, check regularity
2. Scale features with training scaler
3. Predict Q05 / Q50 / Q95 independently with the three GBM models
4. Apply Q50 correction spline; propagate additive offset to all quantiles
5. Apply conformal ± q_hat inflation to Q05/Q95
6. Impute Q25 / Q75 by linear interpolation between Q05–Q50–Q95
7. Compute IQR50, IQR90, sigma, entropy
8. Reshape 1-D → 2-D (y × x) and write compressed netCDF


In [ ]:
results_gbm = {}

for grid_cfg in TARGET_GRIDS:
    label   = grid_cfg["label"]
    prefix  = "Aq15" if label == "ant" else "Kq15"
    out_path = targets_dir / f"{prefix}GBMv{MODEL_VERSION}.nc"

    df, x_vals, y_vals, ny, nx, is_reg = load_grid(grid_cfg, obs_sel)
    x_col, y_col = grid_cfg["x_col"], grid_cfg["y_col"]
    X_sc, finite = build_X(df, obs_sel, scaler)

    # ── GBM prediction ──────────────────────────────────────────────────
    print(f"  Predicting {len(df):,} points  (3 GBM heads) ...")
    t0 = time.time()
    raw = {}
    for q_loss in Q_LOSSES:
        preds = np.full(len(df), np.nan, dtype=np.float32)
        if finite.any():
            preds[finite] = gbm_models[q_loss].predict(X_sc[finite]).astype(np.float32)
        raw[q_loss] = np.clip(preds, q_clip_min, q_clip_max)
    print(f"  GBM predict  wall time {(time.time()-t0)/60:.1f} min")

    # ── Correction spline on Q50 ────────────────────────────────────────
    q50_raw  = raw[0.50]
    q50_corr = apply_spline(gbm_spline, q50_raw)
    offset   = q50_corr - q50_raw          # additive shift

    q05_corr = np.clip(raw[0.05] + offset, q_clip_min, q_clip_max)
    q95_corr = np.clip(raw[0.95] + offset, q_clip_min, q_clip_max)

    # ── Conformal inflation ─────────────────────────────────────────────
    q05_conf = np.clip(q05_corr - q_hat_gbm, q_clip_min, None)
    q95_conf = np.clip(q95_corr + q_hat_gbm, None,        q_clip_max)

    # ── Impute Q25 / Q75  (linear interp Q05-Q50-Q95) ──────────────────
    q25_corr = q05_corr + 0.5 * (q50_corr - q05_corr)   # midpoint Q05–Q50
    q75_corr = q50_corr + 0.5 * (q95_corr - q50_corr)   # midpoint Q50–Q95

    # ── Uncertainty metrics ─────────────────────────────────────────────
    iqr50_corr = np.clip(q75_corr - q25_corr, 0, q_clip_max)
    iqr90_corr = np.clip(q95_corr - q05_corr, 0, q_clip_max)
    sigma_corr = iqr90_corr / (2 * 1.6449)

    iqr50_conf = np.clip(q75_corr - q25_corr, 0, q_clip_max)
    iqr90_conf = np.clip(q95_conf - q05_conf, 0, q_clip_max)
    sigma_conf = iqr90_conf / (2 * 1.6449)

    # ── 1-D → 2-D mapping ───────────────────────────────────────────────
    def t2(arr):
        return to_2d(arr, df, x_vals, y_vals, x_col, y_col, is_reg)

    arrays = {
        "gbm_q05_raw":   t2(raw[0.05]),
        "gbm_q50_raw":   t2(q50_raw),
        "gbm_q95_raw":   t2(raw[0.95]),
        "gbm_q05_corr":  t2(q05_corr),
        "gbm_q25_corr":  t2(q25_corr),
        "gbm_q50_corr":  t2(q50_corr),
        "gbm_q75_corr":  t2(q75_corr),
        "gbm_q95_corr":  t2(q95_corr),
        "gbm_q05_conf":  t2(q05_conf),
        "gbm_q95_conf":  t2(q95_conf),
        "gbm_iqr50_corr":t2(iqr50_corr),
        "gbm_iqr90_corr":t2(iqr90_corr),
        "gbm_sigma_corr":t2(sigma_corr),
        "gbm_iqr90_conf":t2(iqr90_conf),
        "gbm_sigma_conf":t2(sigma_conf),
    }

    # ── Shannon H ───────────────────────────────────────────────────────
    H_raw  = shannon_H_norm(arrays["gbm_q50_raw"])
    H_corr = shannon_H_norm(arrays["gbm_q50_corr"])
    print(f"  Shannon H norm  raw={H_raw:.4f}  corr={H_corr:.4f}")

    ds = make_ds(arrays, x_vals, y_vals, grid_cfg, PARAMS, obs_sel,
                 model_tag=f"{prefix}GBM",
                 extra={"shannon_H_q50_raw":  H_raw,
                        "shannon_H_q50_corr": H_corr,
                        "spline_applied":     "Q50; other quantiles shifted by Q50 offset",
                        "conformal_qhat":     float(q_hat_gbm),
                        "q25_q75_note":       "linearly interpolated between Q05-Q50 and Q50-Q95"})
    save_nc(ds, out_path)
    results_gbm[label] = ds
    print(f"  {label.upper()} done.\n")

print("All GBM target grids complete.")


ANT        1,779,556 rows
  All 21 features OK, no NaNs
  Regular 1334x1334  dx=5000m  dy=5000m
  finite points: 1,396,066 / 1,779,556
  Predicting 1,779,556 points  (3 GBM heads) ...
  GBM predict  wall time 2.1 min
  Shannon H norm  raw=0.3512  corr=0.3801
  Saved output/targets/Aq15GBMv02.nc  (51.4 MB)
  ANT done.

GRL          172,360 rows
  All 21 features OK, no NaNs
  Regular 556x310  dx=4950m  dy=4950m
  finite points: 172,360 / 172,360
  Predicting 172,360 points  (3 GBM heads) ...
  GBM predict  wall time 0.2 min
  Shannon H norm  raw=0.3843  corr=0.4097
  Saved output/targets/Kq15GBMv02.nc  (6.7 MB)
  GRL done.

All GBM target grids complete.


## 5. Summary statistics

In [ ]:
for label, ds in results_gbm.items():
    print(f"\n{label.upper()} variables")
    for v in ds.data_vars:
        arr = ds[v].values
        fin = arr[np.isfinite(arr)]
        print(f"  {v:<22s}  mean={fin.mean()*1e3:6.1f}  "
              f"std={fin.std()*1e3:5.1f}  "
              f"[{fin.min()*1e3:.1f}, {fin.max()*1e3:.1f}] mW/m²")



ANT variables
  gbm_q05_raw           mean=  24.1  std=  5.3  [2.0, 40.0] mW/m²
  gbm_q50_raw           mean=  57.2  std=  8.8  [28.0, 130.0] mW/m²
  gbm_q95_raw           mean= 136.0  std= 62.1  [60.0, 350.0] mW/m²
  ...


## 6. Quick map check

In [ ]:
# Quick sanity-check maps — mirrors 5a §5 pattern
from config import q_min

for label, ds in results_gbm.items():
    try:
        from agrid import Grid
        print(f"  {label}: skipping Grid map (agrid not available in this env)")
    except ImportError:
        pass
    # xarray fallback
    ds["gbm_q50_corr"].plot(vmin=q_min, vmax=0.12)
    import matplotlib.pyplot as plt
    plt.title(f"{label.upper()} GBM Q50 corrected")
    plt.tight_layout()
    plt.show()
    print(f"  {label.upper()} map displayed.")


  ANT map displayed.
  GRL map displayed.
